In [1]:
import pandas as pd
from scipy import stats
import numpy as np
from pandas.api.types import CategoricalDtype
import re
from helpers import get_renamed_blocks, get_renamed_blocks3

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
#STATES = ['in Betrieb', 'Gesetzlich an Stilllegung gehindert', 'Netzreserve',  'Sicherheitsbereitschaft', 'Sonderfall', 'vorläufig stillgelegt', 'stillgelegt', 'Kohlestromvermarktungsverbot']
#STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'KVBG','stillgelegt', 'vorläufig stillgelegt']
ACTIVE_STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'vorläufig stillgelegt']
ENERGIES = ["Kernenergie", "Braunkohle", "Steinkohle", "Erdgas", "Mineralölprodukte", "Abfall", "Biomasse", ""]

In [6]:
bm = pd.read_csv("../basic/inspire_prtr_mapper.csv", engine="python")
plr = pd.read_excel("../data/Kraftwerksliste.xlsx", sheet_name=1, skiprows=10, decimal=',')
pl0 = pd.read_excel("../data/Kraftwerksliste.xlsx", skiprows=11, decimal=',')
prtr = pd.read_excel("../data/2023-12-08_PRTR-Deutschland_Freisetzungen.xlsx")

In [7]:
bm['PRTR_Kennnummer'] = bm['PRTR_Kennnummer'].apply(lambda x: str(x).replace('/', '_'))
bm['InspireID_Betrieb'] = bm['InspireID_Betrieb'].apply(lambda x: str(x).replace('/', '_'))

In [8]:
#bm.loc[bm.plantid.str.contains("_")]

In [9]:
pl0_bak = pl0.copy()
plr_bak = plr.copy()

In [10]:
list(pl0_bak)

['Unnamed: 0',
 'MaStR-Nr. der Stromerzeugungseinheit',
 'Anlagenbetreiber',
 'Anzeige-Name der Stromerzeugungseinheit',
 'PLZ der Einheit',
 'Ort der Einheit',
 'Straße der Einheit',
 'Hausnummer der Einheit',
 'Bundesland der Einheit',
 'Datum der erstmaligen Inbetriebnahme der Einheit',
 'Jahr der Inbetriebnahme der Einheit',
 'Kraftwerksstatus der Einheit',
 'Energieträger',
 'Hauptbrennstoff',
 'Speichertechnologie',
 'Auswertung Energieträger',
 'Wärmeauskopplung (KWK)\n(ja/nein)',
 'Ist die Stromerzeugungseinheit ein Bestandteil eines Grenzkraftwerkes?',
 'Bruttoleistung in MW',
 'Nettonennleistung (elektrische Wirkleistung) in MW',
 'Ist die Stromerzeugungseinheit ein Bestandteil eines Grenzkraftwerkes?: ja \nNettonennleistung der Einspeisung in ein deutsches Netz:',
 'Technologie der Stromerzeugung',
 'Volleinspeisung oder Teileinspeisung?',
 'Anschlussnetzbetreiber',
 'Spannungsebene']

In [11]:
list(plr_bak)

['Unnamed: 0',
 'MaStR-Nr. der Stromerzeugungseinheit',
 'Anzeige-Name der Stromerzeugungseinheit',
 'PLZ der Einheit',
 'Ort der Einheit',
 'Straße der Einheit',
 'Hausnummer der Einheit',
 'Bundesland der Einheit',
 'Datum der erstmaligen Inbetriebnahme der Einheit (Datum/Jahr)',
 'Datum der endgültigen Stilllegung der Einheit (Datum/Jahr)',
 'Kraftwerksstatus der Einheit',
 'Energieträger',
 'Hauptbrennstoff',
 'Speichertechnologie',
 'Auswertung Energieträger',
 'Wärmeauskopplung (KWK)\n(ja/nein)',
 'Bruttoleistung in MW',
 'Nettonennleistung (elektrische Wirkleistung) in MW',
 'Technologie der Stromerzeugung',
 'Volleinspeisung oder Teileinspeisung?',
 'Anschlussnetzbetreiber',
 'Spannungsebene',
 'Unnamed: 22']

In [12]:
cols = [0,12,14,17,20,21]
plt = get_renamed_blocks(pl0)
plq = plt.drop(plt.columns[cols],axis=1)
cols2 = [0,11,13,19,20,22]
plr0a = get_renamed_blocks3(plr)
plr0b = plr0a.drop(plr0a.columns[cols2],axis=1)

In [13]:
#plr0a.sort_values('power')

In [14]:
#list(plq)
#list(plr0b)

In [15]:
list(plq)

['blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'initialopYear',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'fullsupply',
 'TSO',
 'voltagelevel']

In [16]:
list(plr0b)

['blockid',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'endop',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'tech',
 'voltagelevel']

In [17]:
plq0a = plq[~plq['blockid'].isin(["SEE9-Dummy-nicht_EEG", "SEE9-Dummy-EEG", "SEE9-Dummy-EE"])] # remove non-conventional facilities
plq0b = plq0a.sort_values(['power'], ascending=False)

In [18]:
#plq0b

In [19]:
plq0a.insert(10, "endop", np.nan)
plr0b.insert(1, "company", np.nan)

In [20]:
plr0b.insert(9, "initialopYear", np.nan)

In [21]:
list(plr0b)

['blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'initialopYear',
 'endop',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'tech',
 'voltagelevel']

In [22]:
list(plq0a)

['blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'initialopYear',
 'endop',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'fullsupply',
 'TSO',
 'voltagelevel']

In [23]:
plq0a.shape

(2122, 20)

In [24]:
plr0b.shape

(299, 19)

In [25]:
plq0a.shape

(2122, 20)

In [26]:
plr0b.shape

(299, 19)

In [27]:
plr0b

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,tech,voltagelevel
0,BNA0302,NaN,Frimmersdorf,41517.0,Frimmersdorf,NaN,NaN,Nordrhein-Westfalen,1957,NaN,2011,Endgültig Stillgelegt 2011 (ohne StA),NaN,Braunkohle,Nein,NaN,129.000,NaN,Hochspannung (HS)
1,BNA0303,NaN,Frimmersdorf,41517.0,Frimmersdorf,NaN,NaN,Nordrhein-Westfalen,1957,NaN,2011,Endgültig Stillgelegt 2011 (ohne StA),NaN,Braunkohle,Nein,NaN,130.000,NaN,Hochspannung (HS)
2,BNA0304,NaN,Frimmersdorf,41517.0,Frimmersdorf,NaN,NaN,Nordrhein-Westfalen,1960,NaN,2011,Endgültig Stillgelegt 2011 (ohne StA),NaN,Braunkohle,Nein,NaN,124.000,NaN,Höchstspannung (HöS)
3,BNA0414,NaN,Westfalen,59071.0,Hamm-Uentrop,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2011,Endgültig Stillgelegt 2011 (ohne StA),NaN,Steinkohle,Nein,NaN,152.000,NaN,Hochspannung (HS)
4,BNA0415,NaN,Westfalen,59071.0,Hamm-Uentrop,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2011,Endgültig Stillgelegt 2011 (ohne StA),NaN,Steinkohle,Nein,NaN,152.000,NaN,Hochspannung (HS)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,SEE954774732035,NaN,Gasmotor 11,66333.0,Völklingen,Saarbrücker Straße,135 - 137,Saarland,2004-06-01 00:00:00,NaN,2025-03-26 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,NaN,Grubengas,Ja,3.047,2.927,Verbrennungsmotor,Mittelspannung
295,SEE956374067388,NaN,Gasmotor 12,66333.0,Völklingen,Saarbrücker Straße,135 - 137,Saarland,2004-06-01 00:00:00,NaN,2025-03-26 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,NaN,Grubengas,Ja,3.047,2.927,Verbrennungsmotor,Mittelspannung
296,SEE978841169494,NaN,Gasmotor 13,66333.0,Völklingen,Saarbrücker Straße,135 - 137,Saarland,2004-06-01 00:00:00,NaN,2025-03-26 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,NaN,Grubengas,Ja,3.047,2.927,Verbrennungsmotor,Mittelspannung
297,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011-02-21 00:00:00,NaN,2025-04-30 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,Verbrennungsmotor,Mittelspannung


In [28]:
pl_combined = pd.concat([plq0a, plr0b])
pl_combined['energysource'] = pl_combined['energysource'].apply(lambda x: str(x).replace('Wärme', 'Erdgas'))

In [29]:
#pl_combined

In [30]:
pl_combined.loc[pl_combined.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2012,Endgültig Stillgelegt 2012 (ohne StA),NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [31]:
pl_combined.shape

(2421, 21)

In [32]:
pl0 = pl_combined.copy()

In [33]:
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()

In [34]:
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]

In [35]:
pl0.loc[pl0["blockid"] == "BNA0645"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech


In [36]:
#pl_debug2.groupby("state").count()

In [37]:
#pl_debug2.groupby("initialop").count()

In [38]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_667488/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [39]:
pl1.loc[pl1.blockid == "SEE966349705634"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
15,SEE966349705634,EnBW Energie Baden-Württemberg AG,Heizkraftwerk Stuttgart-Münster GT 88,70376,Stuttgart,Voltastraße,45,Baden-Württemberg,2025-09-29 00:00:00,2025.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,62.0,56.0,Teileinspeisung (einschließlich Eigenverbrauch),Stuttgart Netze GmbH (SNB947592865054),Hochspannung,NaN


In [40]:
#pl_debug2['initialop'] = pl_debug.initialop.dt.year

In [41]:
#pl_debug['initialop'] = pl_debug['initialop'].to_datetime()

In [42]:
#pl_debug2.groupby("state").count()

In [43]:
#pl_debug2.groupby("initialop").count()

In [44]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_667488/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [45]:
#pl0

In [46]:
#bm

In [47]:
#pl0.groupby("Energieträger").count()

In [48]:
#list(plt)

In [49]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_667488/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [50]:
pl1.loc[pl1.blockid == ""]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech


In [51]:
plt.loc[plt.blockid == "BNA0711"]

,Unnamed: 0,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,state,e1,e2,Speichertechnologie,energysource,chp,border,grosspower,power,gerpower,tech,fullsupply,TSO,voltagelevel


In [52]:
plt = pl1.dropna(subset=["blockid"])

In [53]:
plt = plt.astype({"energysource": 'category'})

In [54]:
plt.loc[plt.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2012,Endgültig Stillgelegt 2012 (ohne StA),NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [55]:
pl2 = pl1.copy()

In [56]:
def to_year(x):
    dt = pd.to_datetime(x)
    return dt.year

In [57]:
def to_year2(x):
    if pd.isna(x):
        return x
    elif isinstance(x, float):
        return x
    elif isinstance(x, int):
        return x
    else:
        ts = pd.Timestamp(ts_input=x)
        #print(ts)
        return int(ts.year)

In [58]:
#blocks.loc[blocks.Energieträger == "Kernenergie"]

In [59]:
pl2

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023-04-05 00:00:00,2023.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001-06-01 00:00:00,2001.0,NaN,In Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025-05-08 00:00:00,2025.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981-01-01 00:00:00,NaN,2025-01-01 00:00:00,endgültig stillgelegt 2025 (nach § 13b EnWG),"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
291,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014-07-31 00:00:00,NaN,2024-08-24 00:00:00,endgültig stillgelegt 2024 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
292,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958-01-01 00:00:00,NaN,2025-07-06 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
297,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011-02-21 00:00:00,NaN,2025-04-30 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [60]:
'''
pl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012
pl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002
pl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970
pl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951
pl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990
pl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953
pl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006
pl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012



pl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D

pl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld
'''

#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace('\n','')
#pl2['Nettoleistung'] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace(';','.')

"\npl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012\npl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002\npl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970\npl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951\npl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990\npl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953\npl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006\npl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012\n\n\n\npl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D\n\npl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld\n"

In [61]:
#pl2[] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
pl2['initialop'] = pl2['initialop'].apply(to_year2)
#pl2['endop'] = pl2['endop'].apply(to_year2)
pl2['power'] = pl2['power'].apply(pd.to_numeric)

In [62]:
pl2.loc[pl2.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963.0,NaN,2012,Endgültig Stillgelegt 2012 (ohne StA),NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [63]:
#bm[pl2.duplicated(['BlockID'], keep=False)].sort_values("BlockID", ascending=False)

In [64]:
#pl2[pl2.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [65]:
#cols = [10,11,12,14,17,18,19]
#pl3 = pl2.drop(pl2.columns[cols],axis=1)

In [66]:
#pl3.groupby('Unternehmen').max()

In [67]:
#pl3

In [68]:
def fix_company(company):
    #print(company)
    company_dict = {"RWE": "RWE AG", "Vattenfall": "Vattenfall GmbH", "Uniper": "Uniper SE", "EnBW": "EnBW AG", "Steag": "Steag GmbH", "Nordzucker": "Nordzucker AG", "Lausitz Energie": "LEAG"}
    for key, value in company_dict.items():
        if key in str(company):
            return value
    return company

In [69]:
def extract_still(x):
    match = re.findall("[0-9]{4}",x)
    return match[0] if match else np.nan

In [70]:
def fix_kwk(x):
    return "Nein" if x == "nein" else "Ja" if x == "ja" else x

In [71]:
pl4 = pl2.copy()

In [73]:
list(pl2.groupby("state").count().reset_index()['state'])

['Endgültig Stillgelegt 2011 (ohne StA)',
 'Endgültig Stillgelegt 2012 (ohne StA)',
 'Endgültig Stillgelegt 2013 (mit StA)',
 'Endgültig Stillgelegt 2013 (ohne StA)',
 'Endgültig Stillgelegt 2014 (mit StA)',
 'Endgültig Stillgelegt 2014 (ohne StA)',
 'Endgültig Stillgelegt 2015 (mit StA)',
 'Endgültig Stillgelegt 2015 (ohne StA)',
 'Endgültig Stillgelegt 2016 (mit StA)',
 'Endgültig Stillgelegt 2016 (ohne StA)',
 'Endgültig Stillgelegt 2017 (mit StA)',
 'Endgültig Stillgelegt 2017 (ohne StA)',
 'Endgültig Stillgelegt 2018 (mit StA)',
 'Endgültig Stillgelegt 2018 (ohne StA)',
 'Endgültig Stillgelegt 2019 (mit StA)',
 'Endgültig Stillgelegt 2019 (ohne StA)',
 'In Betrieb',
 'Kapazitätsreserve aufgrund von § 13e EnWG',
 'Netzreserve aufgrund von KVBG',
 'Netzreserve aufgrund § 13b EnWG',
 'besonderes netztechnisches Betriebsmittel',
 'endgültig stillgelegt 2017 (ohne § 13b EnWG oder KVBG)',
 'endgültig stillgelegt 2018 (ohne § 13b EnWG oder KVBG)',
 'endgültig stillgelegt 2020 (nach § 13b

In [74]:
z = "endgültig stillgelegt 2022 (ohne § 13b EnWG oder KVBG)"

In [75]:
def extract_state(state):
    if state == "In Betrieb":
        return "in Betrieb"
    elif "Endgültig Stillgelegt" in state:
        return "stillgelegt"
    elif "Vorläufig Stillgelegt" in state:
        return "vorläufig stillgelegt"
    elif "an Stilllegung gehindert" in state:
        return "Gesetzlich an Stilllegung gehindert"
    else:
        return state

In [76]:
def extract_state2(state):
    
    states = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
    matches = ['In Betrieb', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve aufgrund § 13b EnWG', 'befristete Strommarktrückkehr', 'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'endgültig stillgelegt 20[0-9]{2} \(nach KVBG\)', r'[E|e]ndgültig [S|s]tillgelegt 20[0-9]{2} \([ohne]|[mit].']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [77]:
def extract_state3(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt']
    matches = ['In Betrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [87]:
def extract_state4(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt', 'stillgelegt']
    matches = ['In Betrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt', r'Endgültig Stillgelegt 20[0-9]{2}']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [90]:
sq = 'Endgültig Stillgelegt 2011 (ohne StA)'

In [91]:
extract_state4(sq)

'stillgelegt'

In [79]:
#pl4

In [94]:
pl4['company'] = pl4['company'].apply(lambda x: fix_company(x))
#pl4['endop'] = pl4['state'].apply(lambda x: extract_still(x))
pl4['state'] = pl4['state'].apply(lambda x: extract_state4(x))
pl4['chp'] = pl4['chp'].apply(lambda x: fix_kwk(x))

In [95]:
list(pl4.groupby("state").count().reset_index()['state'])

['KVBG',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'in Betrieb',
 'stillgelegt',
 'vorläufig stillgelegt']

In [96]:
atest = pl4.groupby("state").count()

In [97]:
#atest

In [98]:
newdf = pl4.copy()

In [99]:
#newdf.dtypes

In [100]:
newdf["state"] = newdf["state"].astype('category')
newdf["state"] = newdf["state"].cat.set_categories(STATES, ordered=True)

In [101]:
newdf["chp"] = newdf["chp"].astype('category')
newdf["chp"] = newdf["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)

In [102]:
pl3 = newdf

In [103]:
atest = newdf.groupby(["power"]).count()
atest.sort_values("initialop", inplace=True, ascending=False)

In [104]:
#print(atest)

In [105]:
atest = pl3.groupby(["state"], observed=False).count()
# atest

In [106]:
# pl3.rename(columns={ pl3.columns[10]: "Energieträger"}, inplace=True)

In [107]:
#bm[bm.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [108]:
#bm2 = bm.drop_duplicates()

In [109]:
#pl3

In [110]:
#bm

In [111]:
pl4 = bm.merge(pl3, how="right", left_on="sseid", right_on="blockid")

In [112]:
pl4.loc[pl4.blockid == "SEE966349705634"]

,InspireID_Betrieb,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,NaN,NaN,NaN,SEE966349705634,EnBW AG,Heizkraftwerk Stuttgart-Münster GT 88,70376,Stuttgart,Voltastraße,45,Baden-Württemberg,2025.0,2025.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,62.0,56.0,Teileinspeisung (einschließlich Eigenverbrauch),Stuttgart Netze GmbH (SNB947592865054),Hochspannung,NaN


In [113]:
pl4

,InspireID_Betrieb,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,NaN,NaN,NaN,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,2023.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,NaN,NaN,NaN,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,NaN,NaN,NaN,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001.0,2001.0,NaN,in Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,NaN,NaN,NaN,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,NaN,NaN,NaN,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025.0,2025.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025-01-01 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024-08-24 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025-07-06 00:00:00,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025-04-30 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [114]:
#pl4.loc[pd.isnull(pl4['sseid'])]

In [115]:
pl5 = pl4[pd.notnull(pl4['blockid'])]
#pl5 = pl4
pl5a = pl5[pd.notnull(pl5['power'])]
#pl6 =  pl5a.copy() #[pd.notnull(pl5a['plantid'])]
pl6 = pl5a.rename(columns={"InspireID_Betrieb": "plantid"})

In [116]:
pl6

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,NaN,NaN,NaN,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,2023.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,NaN,NaN,NaN,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,NaN,NaN,NaN,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001.0,2001.0,NaN,in Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,NaN,NaN,NaN,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,NaN,NaN,NaN,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025.0,2025.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025-01-01 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024-08-24 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025-07-06 00:00:00,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025-04-30 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [117]:
plnewtest = pl6.dropna(subset="plantid").sort_values("plantid")

In [118]:
pl6

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,NaN,NaN,NaN,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,2023.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,NaN,NaN,NaN,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,NaN,NaN,NaN,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001.0,2001.0,NaN,in Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,NaN,NaN,NaN,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,NaN,NaN,NaN,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025.0,2025.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025-01-01 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024-08-24 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025-07-06 00:00:00,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025-04-30 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [119]:
#plnewtest

In [120]:
#pl6.dtypes

In [121]:
pl6.loc[pl6.blockid == "BNA0711"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
1427,NW300-0326774,06-05-300-0326774,BNA0711,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963.0,NaN,2012,stillgelegt,NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [122]:
pl6['initialop'] = pl6['initialop'].apply(pd.to_numeric)
pl6['endop'] = pl6['endop'].apply(lambda x: to_year2(x))

In [123]:
#pl6.loc[pl6.blockid == "BNA0711"].dtypes

In [124]:
pl6.loc[pl6.blockid == "SEE999977738880"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
1549,SD664-02,664-02,SEE999977738880,SEE999977738880,NaN,IKW Deuben,6682.0,Teuchern,Industriestraße,1,Sachsen-Anhalt,1936.0,NaN,2021.0,stillgelegt,Rohbraunkohlen,Braunkohle,Ja,78.0,67.0,NaN,NaN,Hochspannung,Gegendruckmaschine mit Entnahme


In [125]:
blocks = pl6.copy()
stammdaten = pl6.copy()

In [126]:
plants = blocks.copy()
#plants = plants.dropna(subset=["plantid", "initialop"]) #TODO: remove initialop from here

In [127]:
plants

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,NaN,NaN,NaN,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,2023.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,NaN,NaN,NaN,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,NaN,NaN,NaN,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001.0,2001.0,NaN,in Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,NaN,NaN,NaN,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,NaN,NaN,NaN,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025.0,2025.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025.0,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024.0,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025.0,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025.0,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [128]:
plants["energysource"] = plants["energysource"].astype('category')
plants["energysource"] = plants["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants["state"] = plants["state"].astype('category')
#plants["state"] = plants["state"].cat.set_categories(STATES, ordered=True)
plants["chp"] = plants["chp"].astype('category')
plants["chp"] = plants["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants["federalstate"] = plants["federalstate"].astype('category')
plants["federalstate"] = plants["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [129]:
pl1 = pl0.loc[pl0["energysource"].isin(["Kernenergie", "Erdgas", "Steinkohle", "Braunkohle", "Steinkohle"])]

In [130]:
p_grp = plants.groupby("plantid")

In [131]:
#plants.groupby("Energip_subgrper").max()

In [132]:
plants.groupby("state", observed=False).count()

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
state,,,,,,,,,,,,,,,,,,,,,,,
in Betrieb,807,807,807,1344,1344,1232,1232,1232,1163,1138,1344,1232,1232,0,1210,1344,1248,1344,1344,1222,1232,1232,0
Kapazitätsreserve,12,12,12,16,16,16,16,16,16,15,16,16,16,0,16,16,16,16,16,16,16,16,0
Netzreserve,22,22,22,22,22,22,22,22,22,21,22,22,22,0,20,22,22,22,22,22,22,22,0
bnBm,14,14,14,14,14,14,14,14,14,14,14,14,14,0,13,14,14,14,14,14,14,14,0
KVBG,5,5,5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,0
stillgelegt,225,225,225,269,0,266,268,269,203,184,269,268,0,269,170,269,268,155,269,0,0,251,150
vorläufig stillgelegt,13,13,13,18,15,18,18,18,18,17,18,18,18,0,18,18,18,18,18,18,15,17,0


In [133]:
ACTIVE_STATES

['in Betrieb',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'vorläufig stillgelegt']

In [134]:
#plants.loc[plants['state'] == 'in Betrieb']

In [135]:
#plants.loc[plants['state'].isin(ACTIVE_STATES)].sort_values('power', ascending=False)

In [136]:
p_subgrp = plants.loc[plants['state'].isin(ACTIVE_STATES)].groupby('plantid') # TODO: add 

In [137]:
plants_act = pd.DataFrame()
for plantid, group in p_subgrp:
    entry = {}
    entry["plantid"] = plantid
    entry["activepower"] = group["power"].sum()
    #plants_act = plants_act.append(entry, ignore_index=True)
    plants_act = pd.concat([plants_act, pd.DataFrame([entry])], ignore_index=True)

In [138]:
plants_act

,plantid,activepower
0,06-02-B10117A007,239.000
1,BB16012651,1.957
2,BB16018798,176.000
3,BB23020389,4.700
4,BB23020490,333.500
...,...,...
281,ST18046,75.643
282,TH30013152,123.500
283,TH62013494,28.250
284,TH72012874,60.875


In [139]:
plants_a = pd.DataFrame()
for plantid, group in p_grp:
    #if not plantid == "06-05-300-9046797":
    #    continue
    entry = {}
    #print(entry)
    entry["plantid"] = plantid
    # entry["BlockID"] = group
    try:
        entry["plantname"] = group["plantname"].value_counts().index[0]
    except IndexError:
        entry["plantname"] = np.nan
    entry["federalstate"] = group["federalstate"].value_counts().index[0]
    entry["energysource"] = group["energysource"].min()
    try:
        entry["chp"] = group["chp"].min()
    except TypeError:
        #print("aneror")
        #print(plantid)
        #print(group['KWK'].count())
        entry["chp"] = ""
    entry["latestexpanded"] = group["initialop"].max()
    entry["initialop"] = group["initialop"].min()
    entry["totalpower"] = group["power"].sum()
    entry["state"] = group["state"].min()
    entry["blockcount"] = group["blockid"].count()
    try:
        entry["company"] = group["company"].value_counts().index[0]
    except IndexError:
        ;
    #print(entry)
    #break
    plants_a = pd.concat([plants_a, pd.DataFrame([entry])], ignore_index=True)

In [140]:
plants_a

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company
0,06-02-B10117A007,Tiefstack GuD GT41,Hamburg,Steinkohle,Ja,2009.0,1993.0,239.000,in Betrieb,2,Hamburger Energiewerke GmbH
1,06-05-100-0030723,HKW Elberfeld,Nordrhein-Westfalen,Steinkohle,Ja,1992.0,1992.0,85.000,stillgelegt,1,NaN
2,06-05-100-0431554,KW Voerde,Nordrhein-Westfalen,Steinkohle,Nein,1985.0,1982.0,1390.000,stillgelegt,2,NaN
3,06-05-100-0853075,KW West,Nordrhein-Westfalen,Steinkohle,Nein,1971.0,1971.0,640.000,stillgelegt,2,NaN
4,16-30-31400103005,Heizkraftwerk Gera-Nord,Thüringen,Erdgas,Ja,1996.0,1996.0,74.000,stillgelegt,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...
346,ST18046,Kraftwerk ZI DT 2,Sachsen-Anhalt,Erdgas,Ja,2017.0,1980.0,75.643,in Betrieb,8,K+S Minerals and Agriculture GmbH
347,TH30013152,DT,Thüringen,Erdgas,Ja,2022.0,1999.0,123.500,in Betrieb,5,SWE Energie GmbH
348,TH62013494,Kraftwerk UB DT1,Thüringen,Erdgas,Ja,2016.0,1964.0,28.250,in Betrieb,3,K+S Minerals and Agriculture GmbH
349,TH72012874,Gasmotor 1 Jena,Thüringen,Erdgas,Ja,2022.0,2022.0,60.875,in Betrieb,5,TEAG Thüringer Energie AG


In [141]:
int_list = ['blockcount', 'initialop', 'latestexpanded']
for column in int_list:
    plants_a[column] = plants_a[column].astype(int)
#plant['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)

In [142]:
#plants_a.dtypes

In [143]:
plants_a["energysource"] = plants_a["energysource"].astype('category')
plants_a["energysource"] = plants_a["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants_a["state"] = plants_a["state"].astype('category')
plants_a["state"] = plants_a["state"].cat.set_categories(STATES, ordered=True)
plants_a["chp"] = plants_a["chp"].astype('category')
plants_a["chp"] = plants_a["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants_a["federalstate"] = plants_a["federalstate"].astype('category')
plants_a["federalstate"] = plants_a["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [144]:
plants_a.loc[plants_a.plantid == "666-999"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [145]:
plants_a.loc[plants_a.plantid == "06-00176010435"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [146]:
plants_a.loc[plants_a.plantid == "06-05-900-0865327"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [147]:
#plants_a = plants.drop_duplicates(subset="KraftwerkID")

In [148]:
blocks.loc[blocks.sseid == "SEE930982693153"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
832,NW300-9046030,06-05-300-9046030,SEE930982693153,SEE930982693153,Knapsack Power GmbH & Co. KG,Knapsack I - Gasturbine GT11,50354,Hürth,Industriestraße,300,Nordrhein-Westfalen,2007.0,2007.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,308.0,308.0,Volleinspeisung,Amprion GmbH (SNB976890256486),Höchstspannung,NaN


In [149]:
column_titles = ['plantid', 'plantname', 'federalstate','energysource', 'chp', 'latestexpanded', 'initialop', 'totalpower', 'state', 'blockcount', 'company']
plants_b = plants_a.reindex(columns=column_titles)

In [150]:
blocks[blocks.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
798,NaN,NaN,NaN,SEE988996320985,Caterpillar Energy Solutions,P31,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,1994.0,1994.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,1.840000,1.840000,Teileinspeisung (einschließlich Eigenverbrauch),MVV Netze GmbH (SNB985472799266),Mittelspannung,NaN
799,NaN,NaN,NaN,SEE988996320985,Caterpillar Energy Solutions,P31,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,1994.0,1994.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,1.840000,1.840000,Teileinspeisung (einschließlich Eigenverbrauch),MVV Netze GmbH (SNB985472799266),Mittelspannung,NaN
1364,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Nordrhein-Westfalen,NaN,NaN,NaN,in Betrieb,NaN,Mineralölprodukte,NaN,67.851410,66.005550,NaN,NaN,NaN,NaN
1378,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Mecklenburg-Vorpommern,NaN,NaN,NaN,in Betrieb,NaN,Steinkohle,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
1389,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Berlin,NaN,NaN,NaN,in Betrieb,NaN,Erdgas,Ja,5.071500,5.063500,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1339,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Baden-Württemberg,NaN,NaN,NaN,in Betrieb,NaN,Erdgas,NaN,707.209857,686.099822,NaN,NaN,NaN,NaN
1338,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Thüringen,NaN,NaN,NaN,in Betrieb,NaN,Braunkohle,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
1337,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Schleswig-Holstein,NaN,NaN,NaN,in Betrieb,NaN,Braunkohle,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
1336,NaN,NaN,NaN,SEE9-Dummy-nicht_EE,Nicht-EE-Anlagen < 10 MW,NaN,NaN,NaN,NaN,NaN,Sachsen-Anhalt,NaN,NaN,NaN,in Betrieb,NaN,Braunkohle,NaN,11.268000,8.873000,NaN,NaN,NaN,NaN


In [151]:
plants_b.sort_values("initialop", ascending=False, inplace=True)

In [152]:
plants_final = plants_b.loc[:]

In [153]:
#plants_final.dtypes

In [154]:
# plants_final

In [155]:
# stammdaten

In [156]:
#stammdaten

In [157]:
# cols = [0,2,3,9,10,11,12]
stammdaten = pl6.copy()
drop_list = ['plantid', 'energysource', 'initialop', 'chp', 'plantname', 'power', 'state', 'endop', 'company', 'sseid', 'TSO', 'voltagelevel']
stammdaten_dafuq = stammdaten.drop(drop_list, axis=1)
stammdaten = stammdaten_dafuq.copy()

In [158]:
stammdaten_dafuq.sort_values(['blockid'])

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,grosspower,fullsupply,tech
1424,06-08-4380932,BNA0011,79774.0,Albbruck,NaN,NaN,Baden-Württemberg,NaN,Steinkohle,NaN,NaN,NaN
1519,03-07-07244141350,BNA0012d,31061.0,Alfeld,Mühlenmarsch,1,Niedersachsen,NaN,NaN,NaN,NaN,NaN
1446,06-26200010633,BNA0059a,34225.0,Baunatal,NaN,NaN,Hessen,NaN,NaN,NaN,NaN,NaN
1522,06-11-01-1105607,BNA0075,12207.0,Berlin,Ostpreußendamm,61,Berlin,NaN,NaN,NaN,NaN,NaN
1505,06-11-01-1105607,BNA0076,12207.0,Berlin,Ostpreußendamm,61,Berlin,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
773,NaN,SEE999819576335,39126,Magdeburg,Kraftwerk-Privatweg,7,Sachsen-Anhalt,2006.0,"Abfall (Hausmüll, Siedl.abf.)",33.6,Volleinspeisung,NaN
253,06-05-100-0154540,SEE999839188105,40476,Düsseldorf,Rather Straße,51,Nordrhein-Westfalen,2012.0,"Erdgas, Erdölgas",4.3,Teileinspeisung (einschließlich Eigenverbrauch),NaN
1546,06-02-BERZ003800,SEE999848168525,21079.0,Hamburg,Moorburger Schanze,2,Hamburg,NaN,Steinkohlen,860.0,NaN,Kondensationsmaschine mit Entnahme
986,NaN,SEE999966551940,14478,Potsdam,Zum Heizwerk,20,Brandenburg,1996.0,"Erdgas, Erdölgas",42.0,Volleinspeisung,NaN


In [159]:
stmp = pl6.copy()
stammdaten2 = stmp.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [160]:
#stammdaten2

In [161]:
DROP_ST = ['blockid', 'jahr', 'kennnummer','betriebsname','betriebsname_2','plz_x','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST2 = ['jahr', 'kennnummer','betriebsname','betriebsname_2','plz','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST3 = ['jahr', 'kennnummer', 'bundesland', 'flusseinzugsgebiet', "betriebsname", "betriebsname_2", 'taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']


In [162]:
#DROP_ST = ['jahr']

In [163]:
plants_d1 = plants_final.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [164]:
#plants_d1.dtypes

In [165]:
pld2 = plants_d1.drop_duplicates(['plantid'])

In [166]:
pld3 = pld2.drop(DROP_ST3, axis=1)
plants_final1 = pld3

In [167]:
#plants_act

In [168]:
plants_final2 = plants_final1.merge(plants_act, on="plantid", how='left')

In [169]:
plants_final3 = plants_final2.dropna(subset=["plantid"])

In [170]:
plants_final3.groupby("energysource").count()

/tmp/ipykernel_667488/2887887728.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  plants_final3.groupby("energysource").count()


,plantid,plantname,federalstate,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
energysource,,,,,,,,,,,,,,,,,,,
Kernenergie,8,8,8,8,8,8,8,8,8,0,0,0,0,0,0,0,0,0,0
Braunkohle,28,28,28,28,28,28,28,28,28,23,0,0,0,0,0,0,0,0,23
Steinkohle,63,63,63,63,63,63,63,63,63,43,4,4,4,4,4,4,4,4,43
Erdgas,229,227,229,229,229,229,229,229,229,204,1,1,1,1,1,1,1,1,204
Mineralölprodukte,22,22,22,22,22,22,22,22,22,15,0,0,0,0,0,0,0,0,15
Abfall,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Biomasse,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [171]:
plants_final2[pd.notnull(plants_final2['activepower'])].sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
230,SN70015796,Boxberg Block N,Sachsen,Braunkohle,Ja,2012,1979,2470.000,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.000
322,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.000,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.000
275,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.000,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.000
217,BB45025564,Kraftwerk Jänschwalde Block A,Brandenburg,Braunkohle,Ja,1989,1981,3000.000,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.000
301,BWpf-450-2948214-00000000,GKM,Baden-Württemberg,Steinkohle,Ja,2015,1966,2388.000,in Betrieb,6,Grosskraftwerk Mannheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1983.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,SN60018636,BHKW-G66-KWK-klein,Sachsen,Erdgas,Ja,43067,43067,3.736,in Betrieb,1,Volkswagen Sachsen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.736
65,SD661-80,0050 - STW - BHKW - KWK,Nordrhein-Westfalen,Erdgas,Ja,2017,2007,12.877,in Betrieb,5,Stadtwerke Kempen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.313
25,NW700-0104479,BHKW-G,Nordrhein-Westfalen,Erdgas,Ja,2016,2016,1.999,in Betrieb,1,Westag AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.999
24,NW100-0036701,BHKW BASF F17,Nordrhein-Westfalen,Erdgas,Ja,2016,2016,1.968,in Betrieb,1,BASF Personal Care and Nutrition GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.968


In [172]:
plants_final2.sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
230,SN70015796,Boxberg Block N,Sachsen,Braunkohle,Ja,2012,1979,2470.0,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.0
322,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.0,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.0
275,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.0,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.0
217,BB45025564,Kraftwerk Jänschwalde Block A,Brandenburg,Braunkohle,Ja,1989,1981,3000.0,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.0
301,BWpf-450-2948214-00000000,GKM,Baden-Württemberg,Steinkohle,Ja,2015,1966,2388.0,in Betrieb,6,Grosskraftwerk Mannheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1983.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335,BWpf-450-1479296-00000000,Bestandsanlage_HKW_Aalen,Baden-Württemberg,Erdgas,Ja,1960,1960,15.0,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
336,NW100-0888309,HKW Venator Germany,Nordrhein-Westfalen,Erdgas,Ja,1960,1960,8.9,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
339,NW900-0271161,Shamrock,Nordrhein-Westfalen,Steinkohle,Ja,1957,1957,132.0,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,NW100-0081105,Frimmersdorf,Nordrhein-Westfalen,Braunkohle,Nein,1970,1957,2008.0,stillgelegt,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [173]:
plants_final2[pd.isnull(plants_final2['plantid'])]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower


In [174]:
stammdaten

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,grosspower,fullsupply,tech
0,NaN,SEE915851127786,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,"Erdgas, Erdölgas",22.535,Volleinspeisung,NaN
1,NaN,SEE973601397087,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,NaN,28.000,Volleinspeisung,NaN
2,NaN,SEE901401382521,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001.0,Wärme,5.000,Teileinspeisung (einschließlich Eigenverbrauch),NaN
3,NaN,SEE999106955379,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,"Erdgas, Erdölgas",51.000,Volleinspeisung,NaN
4,NaN,SEE968161544286,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025.0,"Erdgas, Erdölgas",4.500,Volleinspeisung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1683,06-05-900-0080266,SEE972461974179,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,NaN,"Erdgas, Erdölgas",124.500,NaN,Gasturbinen mit Abhitzekessel
1684,06-09-261-0007-0041,SEE909963484639,84030.0,Landshut,Ohmstraße,2,Bayern,NaN,"Erdgas, Erdölgas",2.627,NaN,Verbrennungsmotor
1685,06-05-300-9046797,SEE956973616802,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,NaN,Steinkohlen,18.000,NaN,Gegendruckmaschine ohne Entnahme
1686,661-14,SEE932929596862,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,NaN,"Erdgas, Erdölgas",2.016,NaN,Verbrennungsmotor


In [175]:
#plants_final2.sort_values("activepower", ascending=False)

In [176]:
#stammdaten2.dtypes

In [177]:
#stammdaten3.sort_values(by=['plantid'], ascending=True)

In [178]:
#stammdaten3 = stammdaten2.drop(DROP_ST, axis=1)

In [179]:
#stammdaten4 = stammdaten3.drop_duplicates(['bnaid'])

In [180]:
#stammdaten4

In [181]:
#list(stammdaten2)

In [182]:
'''


"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,
blockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"
662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201

'''


'\n\n\n"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,\nblockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"\n662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201\n\n'

In [183]:
#|664-02|IKW

In [184]:
blocks2 = blocks[['blockid', 'plantid', 'plantname', 'federalstate', 'energysource', 'initialop', 'chp', 'power', 'state', 'endop', 'company']]

In [185]:
blocks2['initialop'] = blocks2['initialop'].astype('Int32')
blocks2['endop'] = blocks2['endop'].astype('Int32')

/tmp/ipykernel_667488/1874890394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['initialop'] = blocks2['initialop'].astype('Int32')
/tmp/ipykernel_667488/1874890394.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['endop'] = blocks2['endop'].astype('Int32')


In [186]:
blocks2.loc[blocks2.blockid == "06-08-2948214"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company


In [187]:
blocks2.loc[blocks2.plantid == "BYS00041"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1158,SEE942366584926,BYS00041,SWM HKW Nord 1 T10,Bayern,Abfall,1991,Ja,18.0,in Betrieb,<NA>,SWM Services GmbH
1159,SEE952372080091,BYS00041,SWM HKW Nord 2 T20,Bayern,Erdgas,1991,Ja,333.0,in Betrieb,<NA>,SWM Services GmbH
1160,SEE998278854237,BYS00041,SWM HKW Nord 3 T30,Bayern,Abfall,1984,Ja,22.0,in Betrieb,<NA>,SWM Services GmbH


In [188]:
blocks2['chp'] = blocks2['chp'].fillna('Nein')

/tmp/ipykernel_667488/2832682086.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['chp'] = blocks2['chp'].fillna('Nein')


In [189]:
list(blocks2)

['blockid',
 'plantid',
 'plantname',
 'federalstate',
 'energysource',
 'initialop',
 'chp',
 'power',
 'state',
 'endop',
 'company']

In [190]:
stammdaten['plz'] = stammdaten['plz'].astype('Int32')

In [191]:
stammdaten.loc[stammdaten.duplicated(subset=['blockid'])]

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,grosspower,fullsupply,tech
799,NaN,SEE988996320985,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,1994.0,"Erdgas, Erdölgas",1.840,Teileinspeisung (einschließlich Eigenverbrauch),NaN
1308,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Bayern,NaN,NaN,49.826,NaN,NaN
1309,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Berlin,NaN,NaN,0.000,NaN,NaN
1310,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Brandenburg,NaN,NaN,1.100,NaN,NaN
1311,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Bremen,NaN,NaN,18.600,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1414,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Saarland,NaN,NaN,0.000,NaN,NaN
1415,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Sachsen,NaN,NaN,0.000,NaN,NaN
1416,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Sachsen-Anhalt,NaN,NaN,0.000,NaN,NaN
1417,NaN,SEE9-Dummy-nicht_EE,<NA>,NaN,NaN,NaN,Schleswig-Holstein,NaN,NaN,0.224,NaN,NaN


In [192]:
stammdaten.drop_duplicates(subset="blockid", inplace=True)
blocks2.drop_duplicates(subset="blockid", inplace=True)
plants_final2.drop_duplicates(subset="plantid", inplace=True)

/tmp/ipykernel_667488/3735086467.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2.drop_duplicates(subset="blockid", inplace=True)


In [193]:
stammdaten2.loc[stammdaten2.blockid == "SEE915851127786"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz_x,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech,jahr,kennnummer,betriebsname,betriebsname_2,betreiber,eigentuemer,plz_y,ort,strasse,hausnr,bundesland,flusseinzugsgebiet,geo_lat_wgs84,geo_long_wgs84,taet_nr,taetigkeit,activity,haupttaetigkeit,branche,sector,nace_id,nace_wirtschaftszweig,nace_sector,stoffgruppe,substances_group,schadstoff,pollutant,umweltkompartiment,releases_to,jahresfracht_freisetzung,versehentliche_freisetzung,schadstoff_schwellenwert,einheit,unit,bestimmungsmethode,determination_method,schutzgrund_fracht,confidential_reason_release,schutzgrund_betrieb,confidential_reason_facility
0,NaN,NaN,NaN,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,2023.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [194]:
stammdaten2 = stammdaten[['blockid', 'plz', 'place', 'street', 'streetnum', 'federalstate']]

In [195]:
blocks2.loc[blocks2.blockid == "SEE966349705634"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
8,SEE966349705634,NaN,Heizkraftwerk Stuttgart-Münster GT 88,Baden-Württemberg,Erdgas,2025,Ja,56.0,in Betrieb,<NA>,EnBW AG


In [196]:
#blocks2.loc['C', 'x'] = "BNA1949"

In [197]:
stammdaten2.to_csv("stammdaten_nh_new.csv", index=False, header=False)
blocks2.to_csv("blocks_nh_2.csv", index=False, header=False)
blocks2.to_csv("blocks_new_nh.csv", index=False, header=False)
plants_final2.to_csv("plants_nh_2.csv", index=False, header=False)
stammdaten2.to_csv("stammdaten.csv", index=False)
blocks2.to_csv("blocks_2.csv", index=False)
plants_final2.to_csv("plants_2.csv", index=False)

In [198]:
#sqlite3 plantwatch.db  "CREATE TABLE plants(plantid TEXT NOT NULL PRIMARY KEY, plantname TEXT, federalstate TEXT, energysource TEXT, chp TEXT, latestexpanded INT, initialop INT, totalpower REAL, state TEXT, blockcount INT,  company TEXT, plz TEXT, place TEXT, street TEXT, number TEXT, latitude REAL, longitude REAL, activepower REAL, energy_2015 INTEGER,  energy_2016 INTEGER,  energy_2017 INTEGER,  energy_2018 INTEGER,  energy_2019 INTEGER, energy_2020 INTEGER, energy_2021 INTEGER, co2_2007 INTEGER,  co2_2008 INTEGER,  co2_2009 INTEGER,  co2_2010 INTEGER,  co2_2011 INTEGER,  co2_2012 INTEGER,  co2_2013 INTEGER,  co2_2014 INTEGER,  co2_2015 INTEGER,  co2_2016 INTEGER,  co2_2017 INTEGER,  co2_2018 INTEGER, co2_2019 INTEGER, co2_2020 INTEGER, FOREIGN KEY (plantid) REFERENCES blocks(plantid) ON DELETE CASCADE);"


In [199]:
blocks2.dropna(subset=['plantid'])

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
10,SEE946736241334,BWpf-450-80920445-00000000,HKW Aalen,Baden-Württemberg,Erdgas,2021,Ja,78.302,in Betrieb,<NA>,Palm Power GmbH & Co. KG
15,SEE902796244388,SD661-59,Turbine 4,Niedersachsen,Erdgas,1970,Ja,11.000,in Betrieb,<NA>,Sappi Alfeld
17,SEE971282697380,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau Block 4 GT A (s...,Baden-Württemberg,Erdgas,1971,Nein,45.000,in Betrieb,<NA>,EnBW AG
18,SEE905257392765,MV60004935,Kesselhaus,Mecklenburg-Vorpommern,Erdgas,1993,Ja,15.000,in Betrieb,<NA>,Cosun Beet Company GmbH & Co. KG
19,SEE998387657606,HE50000213,VW Baunatal Dampfturbine,Hessen,Erdgas,2013,Ja,29.304,in Betrieb,<NA>,VW Kraftwerk GmbH
...,...,...,...,...,...,...,...,...,...,...,...
1683,SEE972461974179,NW900-0080266,Heizkraftwerk Hagen-Kabel H5,Nordrhein-Westfalen,Erdgas,1981,Ja,121.000,stillgelegt,2025,NaN
1684,SEE909963484639,BYS00291,BMW Landshut KWK 3,Bayern,Erdgas,2014,Ja,2.550,stillgelegt,2024,NaN
1685,SEE956973616802,NW300-9046797,LEV KWG G12,Nordrhein-Westfalen,Steinkohle,1958,Ja,17.235,stillgelegt,2025,NaN
1686,SEE932929596862,SD661-14,BHKW Hauffstraße,Baden-Württemberg,Erdgas,2011,Ja,1.968,stillgelegt,2025,NaN


In [200]:
blocks2.shape

(1576, 11)

In [163]:
blocks2.dropna(subset=['plantid']).shape

(1096, 11)

In [164]:
#stammdaten.dropna(subset=['sseid', 'bnaid']).sort_values(by=['sseid'], ascending=False)

In [165]:
stammdaten

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,gerpower,fullsupply,grosspower
0,NaN,SEE915851127786,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
1,06-08-80920445,SEE946736241334,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
2,NaN,SEE930596800480,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
3,NaN,SEE929797382345,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
4,NaN,SEE988046628214,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1991.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1533,03-01-01010157480,SEE975094128629,38114.0,Braunschweig,Reiherstraße,3,Niedersachsen,NaN,"Erdgas, Erdölgas",NaN,Volleinspeisung,22.400
1534,NaN,SEE926418118239,49479.0,Ibbenbüren,Groner Allee,76,Nordrhein-Westfalen,NaN,"Erdgas, Erdölgas",NaN,Teileinspeisung (einschließlich Eigenverbrauch),1.853
1535,06-08-2142201,SEE937157344278,74399.0,Walheim,Mühlstraße,1,Baden-Württemberg,NaN,Steinkohlen,NaN,Volleinspeisung,107.000
1536,06-08-4124448,SEE981220191160,77704.0,Oberkirch,Hauptstraße,2,Baden-Württemberg,NaN,Steinkohlen,NaN,Teileinspeisung (einschließlich Eigenverbrauch),20.000


In [166]:
blocks.loc[blocks["blockid"] == "BNA0645"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower


In [167]:
#bm2.to_csv("bpm.csv", index=False)

In [168]:
#blocks

In [169]:
#blocks.groupby('Energieträger').count()

In [170]:
#pl4 = blocks.loc[blocks['Energieträger'] == "Mineralölprodukte"]

In [171]:
#pl4

In [172]:
#pd.set_option('display.max_rows', None)

In [173]:
#pl4.loc[pl4['Energieträger'] == "Mineralölprodukte"].sort_values(["Bundesland", "Unternehmen", "Kraftwerksname"])

In [174]:
#plants_final2

In [175]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1482,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988.0,Nein,1410.0,stillgelegt,2023.0,NaN
1423,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986.0,Nein,1410.0,stillgelegt,2021.0,NaN
1401,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985.0,Nein,1402.0,stillgelegt,2019.0,NaN
1426,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984.0,Nein,1360.0,stillgelegt,2021.0,NaN
1486,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988.0,Nein,1336.0,stillgelegt,2023.0,NaN
1491,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989.0,Nein,1310.0,stillgelegt,2023.0,NaN
1432,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984.0,Nein,1288.0,stillgelegt,2021.0,NaN
1374,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984.0,Nein,1284.0,stillgelegt,2017.0,NaN
1336,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982.0,Nein,1275.0,stillgelegt,2015.0,NaN
393,SEE993592183001,NW100-0248923,Neurath G,Nordrhein-Westfalen,Braunkohle,2012.0,NaN,1060.0,in Betrieb,NaN,RWE AG


In [293]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1482,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988,Nein,1410.0,stillgelegt,2023,NaN
1423,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986,Nein,1410.0,stillgelegt,2021,NaN
1401,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985,Nein,1402.0,stillgelegt,2019,NaN
1426,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984,Nein,1360.0,stillgelegt,2021,NaN
1486,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988,Nein,1336.0,stillgelegt,2023,NaN
1491,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989,Nein,1310.0,stillgelegt,2023,NaN
1432,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984,Nein,1288.0,stillgelegt,2021,NaN
1374,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984,Nein,1284.0,stillgelegt,2017,NaN
1336,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982,Nein,1275.0,stillgelegt,2015,NaN
393,SEE993592183001,NW100-0248923,Neurath G,Nordrhein-Westfalen,Braunkohle,2012,NaN,1060.0,in Betrieb,<NA>,RWE AG
